In [ ]:
import csv

with open("자동차_판매_데이터.csv", "r", encoding = 'utf-8-sig') as file:
    reader = csv.DictReader(file)
    sales_data = list(reader)

#### 4-1. 판매 거래를 표현하는 'sales' 클래스 만들기

In [22]:
class CarSale:
    def __init__(self, sale_id, manufacturer, model, final_sale_amount, delivery_days,
                 stock_status, is_returned):
        self.sale_id = sale_id
        self.manufacturer = manufacturer
        self.model = model
        self.final_sale_amount = final_sale_amount
        self.delivery_days = delivery_days
        self.stock_status = stock_status
        self.is_returned = is_returned

    def get_amount_category(self):
        if self.final_sale_amount >= 70_000_000:
            return "고가"
        elif self.final_sale_amount >= 40_000_000:
            return "중가"
        else:
            return "저가"

    def needs_delivery_check(self):
        if self.delivery_days >= 20 and self.stock_status != '출고완료':
            return True
        else:
            return False

    def is_high_value_return(self, min_amount=70_000_000):
        if self.final_sale_amount >= min_amount and self.is_returned == 'Y':
            return True
        else:
            return False

test = CarSale("TEST001", "현대", "테스트차량", 75000000, 25, "재고있음", "Y")

print(test.sale_id)  # Output: TEST001
print(test.manufacturer)  # Output: 현대
print(test.model)  # Output: 테스트차량
print(test.get_amount_category())  # Output: 고가
print(test.needs_delivery_check())  # Output: True
print(test.is_high_value_return())  # Output: True

TEST001
현대
테스트차량
고가
True
True


#### 4-2. 여러 판매 객체를 관리하는 'CarSalesManager' 클래스 만들기

In [25]:
class CarSalesManager:
    def __init__(self, sales):
        self.sales = sales # List of CarSale instances

    def count_by_amount_category(self):
        categories = {"고가": 0, "중가": 0, "저가": 0}

        for sale in self.sales:
            category = sale.get_amount_category()
            categories[category] += 1

        return categories

    def find_delivery_check_sales(self):
        delivery_check_sales = []

        for sale in self.sales:
            if sale.needs_delivery_check():
                delivery_check_sales.append(sale)

        return delivery_check_sales

    def find_high_value_returns(self):
        high_value_returns = []

        for sale in self.sales:
            if sale.is_high_value_return():
                high_value_returns.append(sale)

        return high_value_returns

    def get_summary(self):
        summary = {
            "total_sales": len(self.sales) }
        return summary

cars = []

for row in sales_data:
    car = CarSale(
        row['SaleID'],
        row['Manufacturer'],
        row['Model'],
        int(row['FinalSaleAmount']),
        int(row['DeliveryDays']),
        row['StockStatus'],
        row['IsReturned']
    )

    cars.append(car)

manager = CarSalesManager(cars)

print('전체 거래 수:', len(manager.sales))
print('판매금액 구간별 거래 수:', manager.count_by_amount_category())
print('배송 확인 대상 거래 수:', len(manager.find_delivery_check_sales()))
print('배송 확인 대상 상위 5개 거래:', [sale.sale_id for sale in manager.find_delivery_check_sales()[:5]])
print('고액 반품 거래 수:', len(manager.find_high_value_returns()))
print('고액 반품 거래 상위 5개 거래:', [sale.sale_id for sale in manager.find_high_value_returns()[:5]])

전체 거래 수: 500
판매금액 구간별 거래 수: {'고가': 71, '중가': 135, '저가': 294}
배송 확인 대상 거래 수: 48
배송 확인 대상 상위 5개 거래: ['S2025010207', 'S2025010051', 'S2025010165', 'S2025020062', 'S2025020216']
고액 반품 거래 수: 3
고액 반품 거래 상위 5개 거래: ['S2025050451', 'S2025100176', 'S2025110197']


#### 4-3. 상속을 활용한 'PremiumCarSalesManager' 클래스 만들기

In [26]:
class PremiuumCarSalesManager(CarSalesManager):
    def __init__(self, sales, premium_threshold=70_000_000):
        super().__init__(sales)
        self.premium_threshold = premium_threshold

    def find_premium_sales(self):
        premium_sales = []

        for sale in self.sales:
            if sale.final_sale_amount >= self.premium_threshold:
                premium_sales.append(sale)

        return premium_sales

    def get_premium_summary(self):
        premium_sales = self.find_premium_sales()
        summary = {
            "premium_count": len(premium_sales),
            "premium_total_amount": 0
        }

        for sale in premium_sales:
            summary["premium_total_amount"] += sale.final_sale_amount

        return summary

premium_manager = PremiuumCarSalesManager(cars)
print('고액 거래 기준:', premium_manager.premium_threshold)
print('고액 거래 수:', premium_manager.get_premium_summary()["premium_count"])
print('고액 거래 총 판매금액:', premium_manager.get_premium_summary()["premium_total_amount"])
print('고액 거래 상위 5개 거래:', [sale.sale_id for sale in premium_manager.find_premium_sales()[:5]])

고액 거래 기준: 70000000
고액 거래 수: 71
고액 거래 총 판매금액: 9099300000
고액 거래 상위 5개 거래: ['S2025010064', 'S2025010264', 'S2025010093', 'S2025010135', 'S2025010483']
